# LEMON Respiration EDA

Run all cells to review the published tables and seven figures, and regenerate the Pearson and Spearman figure. To reanalyse raw recordings, set `RUN_FROM_RAW = True` and point `RAW_DATA` to the respiration TSV/JSON pairs. The default raw path is the local pilot folder. Use the complete dataset path for a full rerun.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

cwd = Path.cwd()
ROOT = next(
    (
        folder for folder in (cwd, *cwd.parents)
        if (folder / "respiration_eda/lemon_respiration_eda.py").is_file()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from inside the project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RUN_FROM_RAW = False
RAW_DATA = ROOT / "data/lemon/pilot"
CANDIDATES = ROOT / "outputs/lemon/psychology_other/psq_bold_ecg_candidates.csv"
DEMOGRAPHICS = (
    ROOT / "data/lemon/source/Behavioural_Data_MPILMBB_LEMON"
    / "META_File_IDs_Age_Gender_Education_Drug_Smoke_SKID_LEMON.csv"
)
OUTPUT = ROOT / "outputs/lemon/respiration_notebook"

## Data and outputs

The raw run checks breathing signals and scanner timing, estimates breathing rate by Welch and peak intervals, and compares coverage across groups. It writes new outputs to the notebook results folder. The published tables remain available for the default review.

In [ ]:
if RUN_FROM_RAW:
    paths = (RAW_DATA, CANDIDATES, DEMOGRAPHICS)
    missing = [path for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Configure the missing raw inputs: {missing}")
    table_dir = figure_dir = OUTPUT / "raw_run"
    subprocess.run(
        [
            sys.executable, "-m", "respiration_eda.lemon_respiration_eda",
            "--data", str(RAW_DATA),
            "--base", str(CANDIDATES),
            "--demographics", str(DEMOGRAPHICS),
            "--output", str(table_dir),
        ],
        cwd=ROOT,
        check=True,
    )
    correlation_figure = figure_dir / "respiration_correlations.png"
else:
    table_dir = ROOT / "outputs/lemon/respiration_eda"
    figure_dir = ROOT / "outputs/lemon/respiration_figures"
    correlation_figure = OUTPUT / "respiration_correlations.png"

participants = pd.read_csv(table_dir / "participant_eda.csv")
coverage = pd.read_csv(table_dir / "cohort_coverage.csv")
windows = pd.read_csv(table_dir / "minute_windows.csv")
recordings = participants.loc[participants["respiration_available"]]
candidates = participants.loc[participants["psq_bold_ecg_candidate"]]
matched = candidates.loc[candidates["respiration_available"]]

## Summary and scan timing

A possible 657-trigger window is a timing candidate. Scan onset needs independent confirmation before physiological correction.

In [ ]:
starts = matched["candidate_657_starts"]
print(f"Breathing recordings: {len(recordings)}")
print(f"Candidates with breathing data: {len(matched)}/{len(candidates)}")
print(f"Median Welch rate: {recordings['spectrum_bpm'].median():.2f} breaths/min")
print(f"Median absolute method difference: {recordings['method_abs_difference_bpm'].median():.2f} breaths/min")
print(f"Possible scan starts: {(starts == 1).sum()} unique, {(starts > 1).sum()} multiple, {(starts == 0).sum()} absent")
print(f"One-minute windows: {len(windows)}")
display(coverage)

## Figures

Coverage, trigger timing, signal quality, method agreement, within-recording variation, example traces, sensitivity, and correlations.

In [ ]:
if not RUN_FROM_RAW:
    from respiration_eda.lemon_respiration_plots import plot_correlations

    OUTPUT.mkdir(parents=True, exist_ok=True)
    plot_correlations(recordings, OUTPUT)

figure_names = (
    "respiration_coverage",
    "respiration_trigger_timing",
    "respiration_quality",
    "respiration_rate_agreement",
    "respiration_within_recording",
    "respiration_examples",
    "respiration_sensitivity",
)
figure_paths = [figure_dir / f"{name}.png" for name in figure_names]
for path in [*figure_paths, correlation_figure]:
    display(Image(filename=str(path), width=900))